# Module 4 Slide 25: Test an HR assistant built with Amazon Bedrock Flows

**What this demo shows:** how to call an Amazon Bedrock **Flow** from Python and stream its answer. The flow acts as an HR assistant that answers employee questions such as leave policy and how to apply for sick leave.

**Important:** unlike the other notebooks in this repo, this one has **no offline mode**. A Bedrock Flow is a resource you build in your own account first, so there is nothing to simulate locally. The notebook is written so it will *not* fail on import: it only calls AWS when you set `RUN_AWS = True` and supply your own `FLOW_ID`.

## What you must create before running (one time)

1. **A knowledge base** in Amazon Bedrock containing HR policy documents. Sample documents are provided in this repo under `sample_data/hr_policies/`. Upload them to an S3 bucket, then create a Bedrock Knowledge Base pointing at that prefix and sync it.
2. **A Bedrock Flow** (Amazon Bedrock console -> *Flows* -> *Create flow*) with these nodes:
   - A **Flow input** node whose output name is `document`.
   - A **Knowledge Base** node (or a **Prompt** node) that receives the question.
   - A **Flow output** node that returns the answer as `document`.
   - Rename the input node to `InputQuestion` so it matches the code below, or edit `nodeName` in the code to match your flow.
3. **Enable model access** for the model your flow uses (Amazon Bedrock console -> *Model access*).
4. Note the **Flow ID** and, once you create a flow **alias/version**, its **alias ID**. The draft alias is `TSTALIASID`.

## IAM permissions the notebook principal needs

- `bedrock:InvokeFlow` on your flow, plus `bedrock:GetFlow` to inspect it.
- The **flow's service role** (not the notebook principal) needs `bedrock:InvokeModel` on the chosen model and `bedrock:Retrieve` on the knowledge base. You configure that role when you create the flow.

See `setup/create_roles.md` in this repo for ready-to-run AWS CLI commands.

**Cost note:** invoking a flow calls a foundation model and (if used) a knowledge base retrieval, so it incurs charges.

In [ ]:
import boto3
import json

# ---------------------------------------------------------------------------
# Configuration. Fill these in, then set RUN_AWS = True.
# ---------------------------------------------------------------------------
RUN_AWS = False                     # Set True after you create the flow below.
FLOW_ID = "<YOUR_FLOW_ID>"          # Bedrock console -> Flows -> your flow -> Flow ID
FLOW_ALIAS_ID = "TSTALIASID"       # Default draft alias; use your published alias for production
REGION = "<AWS_REGION>"            # The Region that hosts your Bedrock flow, e.g. "us-east-1"
INPUT_NODE_NAME = "InputQuestion"  # Must match the input node name in your flow

client = None
if RUN_AWS:
    if FLOW_ID.startswith("<"):
        raise ValueError("Set FLOW_ID to your real Bedrock flow ID before enabling AWS.")
    client = boto3.client("bedrock-agent-runtime", region_name=REGION)
    print("Ready to call flow", FLOW_ID, "in", REGION)
else:
    print("AWS disabled. Create your Bedrock flow, fill in FLOW_ID and REGION, then set RUN_AWS = True.")

In [ ]:
def invoke_flow(question: str) -> str:
    """Send one question to the Bedrock flow and return the streamed answer."""
    if not RUN_AWS or client is None:
        raise RuntimeError("Enable RUN_AWS and configure FLOW_ID/REGION first.")
    response = client.invoke_flow(
        flowIdentifier=FLOW_ID,
        flowAliasIdentifier=FLOW_ALIAS_ID,
        inputs=[
            {
                "nodeName": INPUT_NODE_NAME,
                "nodeOutputName": "document",
                "content": {"document": question},
            }
        ],
    )

    # The flow streams events; collect the final output and completion status.
    result = ""
    for event in response["responseStream"]:
        if "flowOutputEvent" in event:
            result = event["flowOutputEvent"]["content"]["document"]
        elif "flowCompletionEvent" in event:
            status = event["flowCompletionEvent"]["completionReason"]
            print(f"Flow completed: {status}")
    return result

In [ ]:
# Sample HR questions. These match the sample policy documents in sample_data/hr_policies/.
questions = [
    "How many leaves can I get in one year?",
    "What is the maternity leave policy?",
    "How do I apply for sick leave?",
]

if RUN_AWS:
    for q in questions:
        print(f"Q: {q}")
        answer = invoke_flow(q)
        print(f"A: {answer}")
        print("-" * 60)
else:
    print("AWS disabled. These are the questions the flow will answer once RUN_AWS = True:")
    for q in questions:
        print(" -", q)